# LTX-Video Generation (Kaggle / Colab) - free text/image-to-video

- Official repo: https://github.com/Lightricks/LTX-Video (Apache-2.0 code)
- Model weights: https://huggingface.co/Lightricks/LTX-Video
- Best free model for realistic faces on T4: **ltxv-2b-0.9.8-distilled** (fast)
  or 13B fp8 (higher quality, more VRAM).
- For the newest LTX-2.5 (22B, audio+video) see https://github.com/Lightricks/LTX-2
  (needs 2x T4 or better; Kaggle T4 x2 recommended).

Frames must be `8*N + 1` (e.g. 121, 129, 257). Resolutions divisible by 32.
Use image-to-video with a reference portrait for consistent faces.

In [ ]:
# Step 1: setup
import os, subprocess, sys
if not os.path.isdir("LTX-Video/.git"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Lightricks/LTX-Video.git"], check=True)
os.chdir("LTX-Video")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", ".[inference]"], check=True)
print("setup OK")

In [ ]:
# Step 2: download model weights
import os, subprocess
from huggingface_hub import hf_hub_download

# 2B distilled (fast, ~9GB) - good for T4
MODEL_FILE = "ltxv-2b-0.9.8-distilled.safetensors"
os.makedirs("checkpoints", exist_ok=True)
path = hf_hub_download(repo_id="Lightricks/LTX-Video",
                       filename=MODEL_FILE, local_dir="checkpoints")
print("model at:", path)

# patch pipeline config to point at our downloaded checkpoint
import yaml
cfg_path = "configs/ltxv-2b-0.9.8-distilled.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint"] = os.path.abspath(path)
with open("configs/ltxv-2b-0.9.8-distilled.local.yaml", "w") as f:
    yaml.safe_dump(cfg, f)
print("config patched")

In [ ]:
# Step 3: Text-to-video
PROMPT = ("A young woman with wavy brown hair stands in golden evening light on a rooftop, "
          "the wind moving her hair; she smiles softly, camera slowly orbiting her.")
HEIGHT, WIDTH, FRAMES = 704, 1216, 121   # 121 frames = 8*15+1

cmd = [sys.executable, "inference.py",
       "--prompt", PROMPT,
       "--height", str(HEIGHT), "--width", str(WIDTH),
       "--num_frames", str(FRAMES),
       "--seed", "42",
       "--pipeline_config", "configs/ltxv-2b-0.9.8-distilled.local.yaml"]
subprocess.run(cmd, check=True)
print("output: outputs/ ... (see notebook outputs)")

In [ ]:
# Step 4: Image-to-video (BEST for face consistency)
# Upload a portrait photo; the model keeps the person's identity.
from IPython.display import display
from ipywidgets import FileUpload
import time, os

up = FileUpload(accept="image/*", multiple=False)
display(up)
for _ in range(300):
    if up.value:
        break
    time.sleep(0.5)
fn = next(iter(up.value))
IMG = os.path.abspath(fn)
with open(IMG, "wb") as f:
    f.write(up.value[fn]["content"])
print("image:", IMG)

PROMPT = "The person from the image looks at the camera, smiles, and slowly turns their head."
cmd = [sys.executable, "inference.py",
       "--prompt", PROMPT,
       "--conditioning_media_paths", IMG,
       "--conditioning_start_frames", "0",
       "--height", "704", "--width", "1216",
       "--num_frames", "121",
       "--seed", "42",
       "--pipeline_config", "configs/ltxv-2b-0.9.8-distilled.local.yaml"]
subprocess.run(cmd, check=True)
print("DONE")